In [2]:
# %% Load post-despacho data
from pathlib import Path
import pandas as pd

# Path to the parquet file, relative to the current notebook
parquet_path = Path("../data/interim/post_despacho_transformed_data") / "post_despacho_transformed.parquet"

# Read the file
df_post_despacho = pd.read_parquet(parquet_path)

# Quick check
df_post_despacho.head()


,timestamp,aes andres,aguacate 1,aguacate 2,aniana vargas 1,aniana vargas 2,baiguaque 1,baiguaque 2,barahona carbon,bersal,...,tavera 1,tavera 2,total eolico,total generado,total hidroelectrica,total programado,total solar,total termico,valdesia 1,valdesia 2
0,2013-01-01 01:00:00,207.0,25.7,26.7,0.0,0.0,0.2,0.0,43.7,NaN,...,0.0,0.0,11.66,1557.26,167.33,1695.1,NaN,1378.27,0.0,0.0
1,2013-01-01 02:00:00,241.0,25.7,25.7,0.0,0.0,0.2,0.0,43.5,NaN,...,0.0,0.0,20.60,1534.19,165.33,1627.8,NaN,1348.26,0.0,0.0
2,2013-01-01 03:00:00,217.0,25.7,25.7,0.0,0.0,0.2,0.0,44.1,NaN,...,0.0,0.0,4.88,1453.91,170.33,1546.7,NaN,1278.70,0.0,0.0
3,2013-01-01 04:00:00,220.0,25.7,25.7,0.0,0.0,0.2,0.0,43.6,NaN,...,0.0,0.0,0.37,1407.16,170.33,1467.4,NaN,1236.46,0.0,0.0
4,2013-01-01 05:00:00,225.0,25.7,25.7,0.0,0.0,0.2,0.0,44.2,NaN,...,0.0,0.0,0.83,1382.51,170.33,1419.0,NaN,1211.35,0.0,0.0


In [3]:
# %% Check for required solar-park columns
cols_to_check = [
    "parque fotovoltaico coastal",
    "parque fotovoltaico cotoperi i",
    "parque fotovoltaico cotoperi ii",
    "parque fotovoltaico cotoperi iii",
    "parque fotovoltaico washington capital 3",
    "parque fotovoltaico marti",
    "parque fotovoltaico washington capital 2",
]

present  = [c for c in cols_to_check if c in df_post_despacho.columns]
missing  = [c for c in cols_to_check if c not in df_post_despacho.columns]

print(f"✅ Present ({len(present)}):", present)
print(f"❌ Missing ({len(missing)}):", missing)


✅ Present (7): ['parque fotovoltaico coastal', 'parque fotovoltaico cotoperi i', 'parque fotovoltaico cotoperi ii', 'parque fotovoltaico cotoperi iii', 'parque fotovoltaico washington capital 3', 'parque fotovoltaico marti', 'parque fotovoltaico washington capital 2']
❌ Missing (0): []


In [4]:
# %% First non-zero timestamp for each solar-park column
import pandas as pd

cols_to_check = [
    "parque fotovoltaico coastal",
    "parque fotovoltaico cotoperi i",
    "parque fotovoltaico cotoperi ii",
    "parque fotovoltaico cotoperi iii",
    "parque fotovoltaico washington capital 3",
    "parque fotovoltaico marti",
    "parque fotovoltaico washington capital 2",
]

records = []
for col in cols_to_check:
    # mask True where we have a real, non-zero value
    mask = df_post_despacho[col].notna() & (df_post_despacho[col] != 0)
    if mask.any():
        first_idx = mask.idxmax()              # → index of the first True
        first_ts = df_post_despacho.at[first_idx, "timestamp"]
        first_val = df_post_despacho.at[first_idx, col]
        records.append({"plant": col,
                        "first_timestamp": first_ts,
                        "first_value": first_val})
    else:
        records.append({"plant": col,
                        "first_timestamp": None,
                        "first_value": None})

first_readings = pd.DataFrame(records)
first_readings


,plant,first_timestamp,first_value
0,parque fotovoltaico coastal,NaT,NaN
1,parque fotovoltaico cotoperi i,2025-05-09 07:00:00,1.18
2,parque fotovoltaico cotoperi ii,2025-05-09 07:00:00,1.08
3,parque fotovoltaico cotoperi iii,2025-05-09 07:00:00,0.61
4,parque fotovoltaico washington capital 3,2025-01-14 17:00:00,7.86
5,parque fotovoltaico marti,NaT,NaN
6,parque fotovoltaico washington capital 2,2025-01-14 17:00:00,7.86


In [5]:
df_post_despacho['reserva caliente'].max()

np.float64(1532.31)

In [6]:
df_post_despacho['reserva fria'].max()

np.float64(1007.8)

In [7]:
import pandas as pd
from pathlib import Path

# ---------- paths ----------
csv_path     = Path("GetPostDespacho_2025-07-07-10.csv")   # adjust if your folder differs
parquet_path = Path("GetPostDespacho_transformed.parquet")

# ---------- load CSV ----------
df = pd.read_csv(csv_path)

# (optional) quick sanity-check
print(df.head())          # first rows
print(df.shape)           # (rows, columns)

# ---------- save to Parquet ----------
df.to_parquet(parquet_path, engine="pyarrow", index=False)

print(f"✔️  Saved {parquet_path} with {df.shape[0]:,} rows and {df.shape[1]} columns")


             timestamp  aes andres  aguacate 1  aguacate 2  aniana vargas 1  \
0  2025-07-07 05:00:00      253.46        0.00       26.24              0.0   
1  2025-07-07 09:00:00      236.23        0.00       26.30              0.0   
2  2025-07-07 07:00:00      238.60        0.00       26.41              0.0   
3  2025-07-07 04:00:00      248.91        0.00       26.31              0.0   
4  2025-07-07 02:00:00      251.83       26.33       26.20              0.0   

   aniana vargas 2  baiguaque 1  baiguaque 2  barahona carbon  bersal  ...  \
0              0.2          0.2          0.0            50.78     0.0  ...   
1              0.2          0.2          0.0            50.42     0.0  ...   
2              0.2          0.2          0.0            49.82     0.0  ...   
3              0.2          0.2          0.0            52.09     0.0  ...   
4              0.2          0.2          0.0            50.61     0.0  ...   

   total generado  total hidroelectrica  total programad

In [10]:
import pandas as pd

df = pd.read_parquet("../data/interim/post_despacho_transformed_data/post_despacho_transformed.parquet")

df.head()

df.info()

df.describe()

df.isna().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109704 entries, 0 to 109703
Columns: 131 entries, timestamp to valdesia 2
dtypes: datetime64[ns](1), float64(130)
memory usage: 109.6 MB


timestamp               0
aes andres              0
aguacate 1              0
aguacate 2              0
aniana vargas 1         0
                    ...  
total programado        0
total solar         58776
total termico           0
valdesia 1              0
valdesia 2              0
Length: 131, dtype: int64

In [11]:
df.head(100)

,timestamp,aes andres,aguacate 1,aguacate 2,aniana vargas 1,aniana vargas 2,baiguaque 1,baiguaque 2,barahona carbon,bersal,...,tavera 1,tavera 2,total eolico,total generado,total hidroelectrica,total programado,total solar,total termico,valdesia 1,valdesia 2
0,2013-01-01 01:00:00,207.0,25.7,26.7,0.0,0.0,0.2,0.0,43.7,NaN,...,0.0,0.0,11.66,1557.26,167.33,1695.1,NaN,1378.27,0.0,0.0
1,2013-01-01 02:00:00,241.0,25.7,25.7,0.0,0.0,0.2,0.0,43.5,NaN,...,0.0,0.0,20.60,1534.19,165.33,1627.8,NaN,1348.26,0.0,0.0
2,2013-01-01 03:00:00,217.0,25.7,25.7,0.0,0.0,0.2,0.0,44.1,NaN,...,0.0,0.0,4.88,1453.91,170.33,1546.7,NaN,1278.70,0.0,0.0
3,2013-01-01 04:00:00,220.0,25.7,25.7,0.0,0.0,0.2,0.0,43.6,NaN,...,0.0,0.0,0.37,1407.16,170.33,1467.4,NaN,1236.46,0.0,0.0
4,2013-01-01 05:00:00,225.0,25.7,25.7,0.0,0.0,0.2,0.0,44.2,NaN,...,0.0,0.0,0.83,1382.51,170.33,1419.0,NaN,1211.35,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2013-01-05 00:00:00,256.0,30.0,30.0,0.0,0.0,0.0,0.0,41.0,NaN,...,0.0,0.0,5.02,1699.00,170.18,1763.0,NaN,1523.80,0.0,0.0
96,2013-01-05 01:00:00,0.0,19.5,0.0,0.0,0.0,0.3,0.0,0.0,NaN,...,0.0,0.0,11.03,1476.55,180.10,1630.2,NaN,1285.42,22.0,25.0
97,2013-01-05 02:00:00,0.0,0.0,0.0,0.0,0.0,0.3,0.0,0.0,NaN,...,0.0,0.0,9.29,1321.66,81.00,1548.9,NaN,1231.37,18.0,0.0
98,2013-01-05 03:00:00,0.0,0.0,0.0,0.0,0.0,0.3,0.0,0.0,NaN,...,0.0,0.0,16.98,1316.17,63.90,1469.7,NaN,1235.29,0.0,0.0


In [2]:
import pandas as pd

df = pd.read_parquet("../data/interim/post_despacho_transformed_data/post_despacho_transformed.parquet")


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109704 entries, 0 to 109703
Columns: 131 entries, timestamp to valdesia 2
dtypes: datetime64[ns](1), float64(130)
memory usage: 109.6 MB


In [7]:
df.head()

,timestamp,aes andres,aguacate 1,aguacate 2,aniana vargas 1,aniana vargas 2,baiguaque 1,baiguaque 2,barahona carbon,bersal,...,tavera 1,tavera 2,total eolico,total generado,total hidroelectrica,total programado,total solar,total termico,valdesia 1,valdesia 2
0,2013-01-01 01:00:00,207.0,25.7,26.7,0.0,0.0,0.2,0.0,43.7,NaN,...,0.0,0.0,11.66,1557.26,167.33,1695.1,NaN,1378.27,0.0,0.0
1,2013-01-01 02:00:00,241.0,25.7,25.7,0.0,0.0,0.2,0.0,43.5,NaN,...,0.0,0.0,20.60,1534.19,165.33,1627.8,NaN,1348.26,0.0,0.0
2,2013-01-01 03:00:00,217.0,25.7,25.7,0.0,0.0,0.2,0.0,44.1,NaN,...,0.0,0.0,4.88,1453.91,170.33,1546.7,NaN,1278.70,0.0,0.0
3,2013-01-01 04:00:00,220.0,25.7,25.7,0.0,0.0,0.2,0.0,43.6,NaN,...,0.0,0.0,0.37,1407.16,170.33,1467.4,NaN,1236.46,0.0,0.0
4,2013-01-01 05:00:00,225.0,25.7,25.7,0.0,0.0,0.2,0.0,44.2,NaN,...,0.0,0.0,0.83,1382.51,170.33,1419.0,NaN,1211.35,0.0,0.0


In [5]:
df2 = pd.read_parquet("GetPostDespacho_transformed.parquet")


In [6]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Columns: 122 entries, timestamp to parque fotovoltaico cotoperi iii
dtypes: float64(121), object(1)
memory usage: 9.7+ KB


In [9]:
df2.head()

,timestamp,aes andres,aguacate 1,aguacate 2,aniana vargas 1,aniana vargas 2,baiguaque 1,baiguaque 2,barahona carbon,bersal,...,total generado,total hidroelectrica,total programado,total solar,total termico,valdesia 1,valdesia 2,parque fotovoltaico cotoperi i,parque fotovoltaico cotoperi ii,parque fotovoltaico cotoperi iii
0,2025-07-07 05:00:00,253.46,0.00,26.24,0.0,0.2,0.2,0.0,50.78,0.0,...,3147.71,131.48,3033.97,0.00,2833.32,0.0,0.0,0.00,0.00,0.00
1,2025-07-07 09:00:00,236.23,0.00,26.30,0.0,0.2,0.2,0.0,50.42,0.0,...,3244.80,125.19,3130.91,569.95,2370.67,0.0,0.0,17.07,16.80,17.48
2,2025-07-07 07:00:00,238.60,0.00,26.41,0.0,0.2,0.2,0.0,49.82,0.0,...,3008.92,135.02,2961.65,41.07,2644.44,0.0,0.0,2.09,2.07,2.09
3,2025-07-07 04:00:00,248.91,0.00,26.31,0.0,0.2,0.2,0.0,52.09,0.0,...,3218.09,131.94,3089.96,0.00,2900.06,0.0,0.0,0.00,0.00,0.00
4,2025-07-07 02:00:00,251.83,26.33,26.20,0.0,0.2,0.2,0.0,50.61,0.0,...,3405.80,208.30,3226.24,0.00,3015.23,0.0,0.0,0.00,0.00,0.00


In [10]:
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------------
# 1)  Load the original parquet
# ------------------------------------------------------------------
src  = Path("../data/interim/post_despacho_transformed_data/"
            "post_despacho_transformed.parquet")
dst  = src.with_name("post_despacho_transformed_parquet_fix.parquet")

df = pd.read_parquet(src)

# ------------------------------------------------------------------
# 2)  Harmonise dtypes
# ------------------------------------------------------------------
# • timestamp  → string (object)  ↔︎  matches the “good” file
# • everything else → float64
df["timestamp"] = df["timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")

num_cols = df.columns.difference(["timestamp"])
df[num_cols] = df[num_cols].astype("float64")

# Optional sanity-check
print(df.dtypes.head())        # timestamp = object, all others float64

# ------------------------------------------------------------------
# 3)  Save the new parquet (no index, pyarrow engine)
# ------------------------------------------------------------------
df.to_parquet(dst, index=False, engine="pyarrow")
print(f"Fixed parquet written to: {dst.resolve()}")


timestamp           object
aes andres         float64
aguacate 1         float64
aguacate 2         float64
aniana vargas 1    float64
dtype: object
Fixed parquet written to: C:\Users\ferna\OneDrive - Universidad APEC - Académico\Documentos\Desktop\11 - Masters\00 - Master AI\99 - Proyecto Final\energy-generation-prediction-dashboard\data\interim\post_despacho_transformed_data\post_despacho_transformed_parquet_fix.parquet


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109704 entries, 0 to 109703
Columns: 131 entries, timestamp to valdesia 2
dtypes: float64(130), object(1)
memory usage: 109.6+ MB


In [12]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Columns: 122 entries, timestamp to parque fotovoltaico cotoperi iii
dtypes: float64(121), object(1)
memory usage: 9.7+ KB


In [2]:
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------------
# 1)  Locate files
# ------------------------------------------------------------------
src = Path("../data/interim/post_despacho_transformed_data/"
           "post_despacho_transformed.parquet")
dst = src.with_suffix(".csv")          # → post_despacho_transformed.csv

# ------------------------------------------------------------------
# 2)  Load Parquet
# ------------------------------------------------------------------
df = pd.read_parquet(src)

# ------------------------------------------------------------------
# 3)  Ensure timestamp is plain text (yyyy-MM-dd HH:mm:ss)
# ------------------------------------------------------------------
if pd.api.types.is_datetime64_any_dtype(df["timestamp"]):
    df["timestamp"] = df["timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")
else:
    # handle nanoseconds-since-epoch integers, if any slipped in
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ns") \
                         .dt.strftime("%Y-%m-%d %H:%M:%S")

# ------------------------------------------------------------------
# 4)  Save as CSV (utf-8, no index, high-throughput quotes=none)
# ------------------------------------------------------------------
df.to_csv(dst, index=False, encoding="utf-8", float_format="%.6f")
print(f"CSV written to: {dst.resolve()}")


CSV written to: C:\Users\ferna\OneDrive - Universidad APEC - Académico\Documentos\Desktop\11 - Masters\00 - Master AI\99 - Proyecto Final\energy-generation-prediction-dashboard\data\interim\post_despacho_transformed_data\post_despacho_transformed.csv
